**PROGETTO N°1:** Algoritmo di Correzione per Motore di Ricerca

**Autore:** Citarelli Federico



**Descrizione:**

Implementare un algoritmo di correzione automatica per un motore di ricerca interno aziendale.
Con l'algoritmo vengono rilevate errori nelle query proposte e corregge l'errore con la parola corretta più probabile basandosi
sulla distanza di Levenshtein calcolata utilizzando un dizionario predefinito di 50 parole.

**Funzione principale:**

    suggest_correction(query, dictionary)
        - Input: query utente e dizionario di parole valide.
        - Output: query corretta o, se già corretta, la query originale.

Sono proposte 10 query per testare l'algoritmo.

In [1]:
## Definisco la variabile DIZIONARIO con all'interno 50 parole che mi serviranno per le correzioni.
DIZIONARIO = {
    1:'rapporto',
    2:'vendite',
    3:'fatturato',
    4:'bilancio',
    5:'clienti',
    6:'cliente',
    7:'contratto',
    8:'proposta',
    9:'documento',
    10:'report',
    11:'meeting',
    12:'agenda',
    13:'progetto',
    14:'sviluppo',
    15:'marketing',
    16:'finanza',
    17:'contabilita',
    18:'reclami',
    19:'assistenza',
    20:'ordine',
    21:'magazzino',
    22:'spedizione',
    23:'fattura',
    24:'pagamento',
    25:'budget',
    26:'analisi',
    27:'performance',
    28:'kpi',
    29:'strategia',
    30:'roadmap',
    31:'risorse',
    32:'formazione',
    33:'personale',
    34:'candidato',
    35:'colloquio',
    36:'stipendio',
    37:'ferie',
    38:'orario',
    39:'sede',
    40:'prodotto',
    41:'servizio',
    42:'prezzo',
    43:'sconto',
    44:'preventivo',
    45:'obiettivo',
    46:'ricavo',
    47:'utile',
    48:'perdita',
    49:'investimento',
    50:'fornitore'
    }


## Funzioni ausiliarie per tutto il processo di analisi e correzione (se necessaria) della query.
def rimuovi_punteggiatura(testo):
    """
    Funzione per rimuovere la punteggiatura dalla frase fonita.

    parametri:
        testo --> è il testo della query che viene fornita dall'utente. è una variabile di tipo stringa.

    output:
        stringa_pulita --> è la query pulita da ogni tipo di punteggiatura.
    """

    stringa_pulita = testo
    punteggiatura = ".,!?;:*+-<>^"

    for carattere in punteggiatura:
        stringa_pulita = stringa_pulita.replace(carattere, "")

    return stringa_pulita

def rimuovi_accento(testo):
    """
    Funzione per rimuovere gli accenti da parole contenute nella query.

    parametri:
        testo --> è il testo della query che viene fornita dall'utente. è una variabile di tipo stringa.

    output:
        testo_senza_accenti --> query in cui le parole con l'accento non hanno più la lettera accentata.
    """

    accenti = {
        'à': 'a', 'è': 'e', 'é': 'e', 'ì': 'i', 'ò': 'o', 'ù': 'u',
        'À': 'A', 'È': 'E', 'É': 'E', 'Ì': 'I', 'Ò': 'O', 'Ù': 'U'
    }

    testo_senza_accenti = ""

    for char in testo:
        testo_senza_accenti += accenti.get(char, char)

    return testo_senza_accenti

def get_numeric_token(token_list):
    """
    Funzione per dividere le parole da caratteri numerici.

    parametri:
        token_list --> è una lista che contine tutti i token della query.

    output:
        numeric_token --> lista con tutti i token numerici
        str_token --> lista con tutti i token NON numerici (parole)
    """

    numeric_token = []
    str_token = []
    for token in token_list:
        if token.isdigit():
            numeric_token.append(token)
        else:
            str_token.append(token)

    return numeric_token, str_token

def levenshtein_distance(s1, s2):
    """
        Funzione per il calcolo della distanza di levenshtein tra due sringhe

        parametri:
            s1 --> stringa 1
            s2 --> stringa 2

        output: dp[m][n] --> la distanza tra le due parole contenute in s1 e s2.

    """

    m = len(s1)
    n = len(s2)

    dp = [[0] * (n+1) for _ in range(m+1)]

    for i in range (m + 1):
        dp[i][0] = i

    for j in range(n + 1):
        dp[0][j] = j

    # compilo la matrice
    for i in range(1, m + 1):
        for j in range(1, n + 1):
            # Se i caratteri sono uguali, il costo è 0, altrimenti è 1 (sostituzione)
            costo = 0 if s1[i - 1] == s2[j - 1] else 1

            # Prendi il minimo tra inserimento, cancellazione e sostituzione
            dp[i][j] = min(dp[i - 1][j] + 1,      # Cancellazione
                           dp[i][j - 1] + 1,      # Inserimento
                           dp[i - 1][j - 1] + costo) # Sostituzione

    return dp[m][n]

def merge_split_tokens(token_list, dictionary):
    """
    Unisce parole spezzate in due token consecutivi se la loro combinazione
    corrisponde (entro una certa soglia di distanza) a una parola valida del dizionario.
    Ad esempio ["Budg", "et"] diventerà "budget"

    parametri:
        - token_list --> è la lista di token NON numerici
        - dictionary --> è il dizionario che contiene le parole con cui si effettua la correzione

    output:
        - merged_tokens --> continene le parole unite (parole finali)
        - merged_map --> contiee una mappaturatra la parola finale e le parole che devono essere unite per creare la parola finale
    """

    merged_tokens = []
    merged_map = {}
    i = 0

    while i < len(token_list):
        if i < len(token_list) - 1:
            combined = token_list[i] + token_list[i + 1]
            for cand in dictionary.values():
                dist = levenshtein_distance(combined, cand)
                norm_dist = dist / max(len(combined), len(cand))
                if dist <= 2 and norm_dist <= 0.4:
                    merged_tokens.append(cand)
                    merged_map[cand] = [token_list[i], token_list[i + 1]]
                    i += 2
                    break
            else:
                merged_tokens.append(token_list[i])
                i += 1
        else:
            merged_tokens.append(token_list[i])
            i += 1

    return merged_tokens, merged_map


def suggest_correction(query, dictionary):

    ## ====== PREPROCESSING ======

    # Convertire tutto in minuscolo
    query_lower = query.lower()

    # Pulire la query dai caratteri di punteggiatura e rimuovi accenti
    query_lower_pulita = rimuovi_punteggiatura(query_lower)
    query_lower_pulita = rimuovi_accento(query_lower_pulita)

    # Rimozione di eventuali spazi aggiuntivi all'inizio e alla fine
    query_lower_pulita = query_lower_pulita.strip()

    # Rimozione degli spazi multipli tra le parole della stringa (TOKENIZZAZIONE)
    query_lower_pulita_split = query_lower_pulita.split() # Restituisce una lista di parole

    # Query senza spazi multipli tra le varie parole
    query_lower_pulita_normalizzata = " ".join(query_lower_pulita_split)

    token_numerici, token_stringa = get_numeric_token(query_lower_pulita_split)

    merged_token_stringa, merged_map = merge_split_tokens(token_stringa, dictionary)

    token_originali = query_lower_pulita_split.copy()

    # Correzione, se necessaria, delle parole sbagliate
    correzioni = {}

    for cand in dictionary.values():
        for token in merged_token_stringa:

            # se la parola è contenuta nel dizionario non applico nessuna correzione
            if token in dictionary.values() and cand in dictionary.values():
                continue

            # calcolo la distanza tra le parole del dizionario e le parole della query, e la norm_dist che mi servirà come ulteriore controllo
            dist = levenshtein_distance(cand, token)
            norm_dist = dist / max(len(token), len(cand))

            # se ho parole molto brevi come kpi, hr, ecc..
            if len(token) <= 3:
                if dist == 1:
                    correzioni[token] = cand
            else:
                if dist == 1:
                    correzioni[token] = cand
                elif dist == 2 and norm_dist <= NORM_THRESHOLD:
                    correzioni[token] = cand
                elif dist > 2 and norm_dist <= NORM_THRESHOLD:
                    correzioni[token] = cand


    ## RICOSTRUZIONE DELLA QUERY

    query_finale_token = []
    skip_next = False

    for i, token in enumerate(token_originali):
        if skip_next:
            skip_next = False
            continue

        # Se è un numero, continuo. Devo solo operare sulle parole
        if token.isdigit():
            query_finale_token.append(token)
            continue

        # Se il token fa parte di una fusione, usa la parola fusa
        fused = None
        for merged_word, original_parts in merged_map.items():
            if i < len(token_originali) - 1 and [token, token_originali[i+1]] == original_parts:
                fused = merged_word
                skip_next = True  # salto il prossimo token perché già fuso
                break

        if fused:
            query_finale_token.append(fused)
            continue

        # Se è stato corretto...
        if token in correzioni:
            query_finale_token.append(correzioni[token])
        else:
            query_finale_token.append(token)


    query_finale = " ".join(query_finale_token)

    return query_finale, query_lower_pulita_normalizzata


#### Main Loop

In [2]:
NORM_THRESHOLD = 0.2

TEST_QUERIES = [
    "raporto vend ite 2023",          # errore singolo + parola separata erroneamente
    "rapporto vendit 2023",           # manca lettera finale
    "fattturato 2022",                # lettera ripetuta
    "bilancioo aziendale",            # errore comune, ultima lettera ripetuta
    "clente nuovo",                   # manca una vocale
    "contrato fornitore",             # errore sulla doppia
    "prgetto sviluppo",               # errore interno
    "analasi marketing",              # lettere invertite
    "report perfomance kpi",          # lettera omessa
    "agenda meeting 2025"             # query corretta (nessuna modifica)
]

print("\n========== TEST AUTOMATICO ==========\n")

for q in TEST_QUERIES:
    final_query, normalized_original_query = suggest_correction(query=q, dictionary=DIZIONARIO)
    print(f"Query originale: {normalized_original_query}")
    print(f"Query corretta:  {final_query}")
    print("-" * 50)


========== TEST AUTOMATICI ==========

Query originale: raporto vend ite 2023
Query corretta:  rapporto vendite 2023
--------------------------------------------------
Query originale: rapporto vendit 2023
Query corretta:  rapporto vendite 2023
--------------------------------------------------
Query originale: fattturato 2022
Query corretta:  fatturato 2022
--------------------------------------------------
Query originale: bilancioo aziendale
Query corretta:  bilancio aziendale
--------------------------------------------------
Query originale: clente nuovo
Query corretta:  cliente nuovo
--------------------------------------------------
Query originale: contrato fornitore
Query corretta:  contratto fornitore
--------------------------------------------------
Query originale: prgetto sviluppo
Query corretta:  progetto sviluppo
--------------------------------------------------
Query originale: analasi marketing
Query corretta:  analisi marketing
-------------------------------------